# Redteam

### Primeiro passo

Vou confirmar o formato real do arquivo.

In [9]:
import gzip

with gzip.open(
    "../data/raw/redteam.txt.gz",
    "rt"
) as f:

    for _ in range(10):
        print(f.readline().strip())

150885,U620@DOM1,C17693,C1003
151036,U748@DOM1,C17693,C305
151648,U748@DOM1,C17693,C728
151993,U6115@DOM1,C17693,C1173
153792,U636@DOM1,C17693,C294
155219,U748@DOM1,C17693,C5693
155399,U748@DOM1,C17693,C152
155460,U748@DOM1,C17693,C2341
155591,U748@DOM1,C17693,C332
156658,U748@DOM1,C17693,C4280


Depois carregamos para Polars

In [10]:
import polars as pl

redteam = pl.read_csv(
    "../data/raw/redteam.txt.gz",
    has_header=False,
    new_columns=[
        "timestamp",
        "user",
        "source_computer",
        "destination_computer"
    ]
)

print(redteam.head())
print(redteam.shape)

shape: (5, 4)
┌───────────┬────────────┬─────────────────┬──────────────────────┐
│ timestamp ┆ user       ┆ source_computer ┆ destination_computer │
│ ---       ┆ ---        ┆ ---             ┆ ---                  │
│ i64       ┆ str        ┆ str             ┆ str                  │
╞═══════════╪════════════╪═════════════════╪══════════════════════╡
│ 150885    ┆ U620@DOM1  ┆ C17693          ┆ C1003                │
│ 151036    ┆ U748@DOM1  ┆ C17693          ┆ C305                 │
│ 151648    ┆ U748@DOM1  ┆ C17693          ┆ C728                 │
│ 151993    ┆ U6115@DOM1 ┆ C17693          ┆ C1173                │
│ 153792    ┆ U636@DOM1  ┆ C17693          ┆ C294                 │
└───────────┴────────────┴─────────────────┴──────────────────────┘
(749, 4)


Limpeza inicial

In [11]:
redteam = (
    redteam
    .with_columns(
        pl.col("user")
        .str.split("@")
        .alias("parts")
    )
    .with_columns([
        pl.col("parts").list.get(0).alias("user"),
        pl.col("parts").list.get(1).alias("domain")
    ])
    .drop("parts")
)

In [12]:
print(
    redteam.select(
        pl.len().alias("events")
    )
)

print(
    redteam.select(
        pl.n_unique("user").alias("users")
    )
)

print(
    redteam.select(
        pl.n_unique("source_computer").alias("source_computers")
    )
)

print(
    redteam.select(
        pl.n_unique("destination_computer").alias("destination_computers")
    )
)

print(
    redteam.select(
        pl.min("timestamp").alias("min_ts"),
        pl.max("timestamp").alias("max_ts")
    )
)

shape: (1, 1)
┌────────┐
│ events │
│ ---    │
│ u32    │
╞════════╡
│ 749    │
└────────┘
shape: (1, 1)
┌───────┐
│ users │
│ ---   │
│ u32   │
╞═══════╡
│ 98    │
└───────┘
shape: (1, 1)
┌──────────────────┐
│ source_computers │
│ ---              │
│ u32              │
╞══════════════════╡
│ 4                │
└──────────────────┘
shape: (1, 1)
┌───────────────────────┐
│ destination_computers │
│ ---                   │
│ u32                   │
╞═══════════════════════╡
│ 301                   │
└───────────────────────┘
shape: (1, 2)
┌────────┬─────────┐
│ min_ts ┆ max_ts  │
│ ---    ┆ ---     │
│ i64    ┆ i64     │
╞════════╪═════════╡
│ 150885 ┆ 2557047 │
└────────┴─────────┘


In [13]:
(
    redteam
    .group_by("source_computer")
    .agg(
        pl.len().alias("events")
    )
    .sort("events", descending=True)
    .head(10)
)

source_computer,events
str,u32
"""C17693""",701
"""C22409""",26
"""C19932""",19
"""C18025""",3


### O que acabamos de descobrir

#### Volume de ataques

`749 eventos maliciosos`

Não é tão pequeno a ponto de ser irrelevante e nem tão grande a ponto de ficar inviável de analisar.

---

#### Usuarios comprometidos

`104 usuários`

no auth simplificado tínhamos:

`11.361 usuários`

Então estamos falando de algo como:

`104 / 11361 ≈ 0,9%`

Ou seja, uma minoria bem pequena.

Exatamente o que esperamos de eventos maliciosos.

---

#### Computadores de origem

``4 computadores``

E um deles domina completamente:

C17693
701 eventos

--- 

#### Hipótese forte

O Red Team provavelmente simulou um cenário assim:

Máquina comprometida
        ↓
Movimentação lateral
        ↓
Vários usuários
        ↓
Vários destinos

Ou seja:

C17693

parece ser um host pivô.

In [ ]:
import polars as pl

redteam = pl.read_csv(
    "../data/raw/redteam.txt.gz",
    has_header=False,
    new_columns=[
        "timestamp",
        "user",
        "source_computer",
        "destination_computer"
    ]
)

print(redteam.head())
print(redteam.shape)

shape: (5, 4)
┌───────────┬────────────┬─────────────────┬──────────────────────┐
│ timestamp ┆ user       ┆ source_computer ┆ destination_computer │
│ ---       ┆ ---        ┆ ---             ┆ ---                  │
│ i64       ┆ str        ┆ str             ┆ str                  │
╞═══════════╪════════════╪═════════════════╪══════════════════════╡
│ 150885    ┆ U620@DOM1  ┆ C17693          ┆ C1003                │
│ 151036    ┆ U748@DOM1  ┆ C17693          ┆ C305                 │
│ 151648    ┆ U748@DOM1  ┆ C17693          ┆ C728                 │
│ 151993    ┆ U6115@DOM1 ┆ C17693          ┆ C1173                │
│ 153792    ┆ U636@DOM1  ┆ C17693          ┆ C294                 │
└───────────┴────────────┴─────────────────┴──────────────────────┘
(749, 4)


### Top usuários atacantes

In [14]:
(
    redteam
    .group_by("user")
    .agg(
        pl.len().alias("events")
    )
    .sort("events", descending=True)
    .head(20)
)

user,events
str,u32
"""U66""",118
"""U3005""",36
"""U737""",33
"""U293""",31
"""U1653""",31
…,…
"""U8601""",13
"""U1480""",12
"""U1145""",12


### Top destinos

In [15]:
(
    redteam
    .group_by("destination_computer")
    .agg(
        pl.len().alias("events")
    )
    .sort("events", descending=True)
    .head(20)
)

destination_computer,events
str,u32
"""C2388""",27
"""C754""",26
"""C2519""",21
"""C1015""",15
"""C395""",15
…,…
"""C5653""",7
"""C8209""",7
"""C1737""",7


In [16]:
redteam = (
    redteam
    .with_columns(
        pl.col("user")
        .str.split("@")
        .list.get(0)
        .alias("user")
    )
)

redteam.write_parquet(
    "../data/bronze/redteam.parquet"
)

In [2]:
import polars as pl

uc = pl.read_parquet("../data/silver/user_computers.parquet")

(
    uc
    .group_by("computer")
    .len()
    .sort("len", descending=True)
    .head(20)
)

computer,len
str,u32
"""C457""",10331
"""C586""",10259
"""C1065""",10254
"""C612""",10236
"""C529""",10225
…,…
"""C585""",5724
"""C743""",5712
"""C2553""",4336


In [3]:
(
    uc
    .filter(~pl.col("computer").str.starts_with("C"))
    .group_by("computer")
    .len()
)

computer,len
str,u32
"""U395""",3
"""U1325""",1
"""U3592""",1
"""U251""",1
"""U2204""",1
…,…
"""U11881""",1
"""U636""",1
"""U11553""",1


In [ ]:
redteam = pl.read_parquet("../data/silver/redteam_events.parquet")

print(redteam["user"].n_unique())

print(
    redteam
    .group_by("user")
    .len()
    .sort("len")
    .head(10)
)